In [0]:
# ==========================================================
# Célula 1 — Configuração
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"

TABELA_IND_UF  = f"{CATALOGO}.{SCHEMA}.indicador_uf"
TABELA_META_UF = f"{CATALOGO}.{SCHEMA}.meta_uf"
TABELA_DEST_UF = f"{CATALOGO}.{SCHEMA}.gold_indicador_uf"

print(f"Indicador UF : {TABELA_IND_UF}")
print(f"Meta UF      : {TABELA_META_UF}")
print(f"Destino      : {TABELA_DEST_UF}")

In [0]:
# ==========================================================
# Célula 2 — Agregação do indicador por UF
#
# Autossuficiente: define a configuração no topo.
# O indicador tem uma linha por (ano, sigla_uf, rede).
# A Gold agrega as redes por média simples.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"
TABELA_IND_UF = f"{CATALOGO}.{SCHEMA}.indicador_uf"

df_uf_ind = spark.table(TABELA_IND_UF)

df_uf_ind_prep = (
    df_uf_ind
    .groupBy("ano", "sigla_uf")
    .agg(
        F.round(F.avg("taxa_alfabetizacao"), 4).alias("taxa_observada"),
        F.round(F.avg("media_portugues"), 4).alias("media_portugues"),
        F.countDistinct("rede").alias("qtd_redes"),
        F.count("*").alias("qtd_registros_rede"),
    )
)

print(f"UFs-ano após agregação: {df_uf_ind_prep.count()}")

print("\nDistribuição de qtd_redes:")
df_uf_ind_prep.groupBy("qtd_redes").count().orderBy("qtd_redes").show()

print("\nCobertura por ano:")
df_uf_ind_prep.groupBy("ano").count().orderBy("ano").show()

In [0]:
# ==========================================================
# Célula 3 — Preparação da meta de UF
#
# Autossuficiente: define a configuração no topo.
# A meta cobre 2023, 2024 e 2025.
# 2023 não tem meta (compromisso começa em 2024).
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"
TABELA_META_UF = f"{CATALOGO}.{SCHEMA}.meta_uf"

df_uf_meta = spark.table(TABELA_META_UF)

df_uf_meta_prep = (
    df_uf_meta
    .select(
        F.col("ano").cast("int").alias("ano"),
        F.col("sigla_uf").cast("string").alias("sigla_uf"),
        F.col("percentual_participacao").cast("double").alias("percentual_participacao"),
        *[F.col(f"meta_alfabetizacao_{a}").cast("double").alias(f"meta_{a}") for a in range(2024, 2031)],
    )
    .withColumn(
        "meta_do_ano",
        F.when(F.col("ano") == 2024, F.col("meta_2024"))
         .when(F.col("ano") == 2025, F.col("meta_2025"))
         .when(F.col("ano") == 2026, F.col("meta_2026"))
         .when(F.col("ano") == 2027, F.col("meta_2027"))
         .when(F.col("ano") == 2028, F.col("meta_2028"))
         .when(F.col("ano") == 2029, F.col("meta_2029"))
         .when(F.col("ano") == 2030, F.col("meta_2030"))
         .otherwise(F.lit(None).cast("double"))
    )
    .dropDuplicates(["ano", "sigla_uf"])
)

print(f"Meta UF preparada: {df_uf_meta_prep.count()}")

print("\nCobertura de meta_do_ano por ano:")
df_uf_meta_prep.groupBy("ano").agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("meta_do_ano").isNotNull(), 1).otherwise(0)).alias("com_meta"),
    F.sum(F.when(F.col("meta_do_ano").isNull(), 1).otherwise(0)).alias("sem_meta"),
).orderBy("ano").show()

In [0]:
# ==========================================================
# Célula 4 — Join e cálculo do status
#
# Autossuficiente: recalcula as duas fontes no topo.
# Full outer join preserva UFs que existem em apenas uma fonte.
# Indicador: 25 UFs | Meta: 27 UFs (inclui DF e RR).
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"
TABELA_IND_UF  = f"{CATALOGO}.{SCHEMA}.indicador_uf"
TABELA_META_UF = f"{CATALOGO}.{SCHEMA}.meta_uf"

# --- Indicador agregado ---
df_uf_ind_prep = (
    spark.table(TABELA_IND_UF)
    .groupBy("ano", "sigla_uf")
    .agg(
        F.round(F.avg("taxa_alfabetizacao"), 4).alias("taxa_observada"),
        F.round(F.avg("media_portugues"), 4).alias("media_portugues"),
        F.countDistinct("rede").alias("qtd_redes"),
        F.count("*").alias("qtd_registros_rede"),
    )
)

# --- Meta preparada ---
df_uf_meta_prep = (
    spark.table(TABELA_META_UF)
    .select(
        F.col("ano").cast("int").alias("ano"),
        F.col("sigla_uf").cast("string").alias("sigla_uf"),
        F.col("percentual_participacao").cast("double").alias("percentual_participacao"),
        *[F.col(f"meta_alfabetizacao_{a}").cast("double").alias(f"meta_{a}") for a in range(2024, 2031)],
    )
    .withColumn(
        "meta_do_ano",
        F.when(F.col("ano") == 2024, F.col("meta_2024"))
         .when(F.col("ano") == 2025, F.col("meta_2025"))
         .when(F.col("ano") == 2026, F.col("meta_2026"))
         .when(F.col("ano") == 2027, F.col("meta_2027"))
         .when(F.col("ano") == 2028, F.col("meta_2028"))
         .when(F.col("ano") == 2029, F.col("meta_2029"))
         .when(F.col("ano") == 2030, F.col("meta_2030"))
         .otherwise(F.lit(None).cast("double"))
    )
    .dropDuplicates(["ano", "sigla_uf"])
)

# --- Join ---
df_gold_uf = (
    df_uf_ind_prep.alias("i")
    .join(df_uf_meta_prep.alias("m"), on=["ano", "sigla_uf"], how="full_outer")
    .select(
        F.coalesce(F.col("i.ano"), F.col("m.ano")).alias("ano"),
        F.coalesce(F.col("i.sigla_uf"), F.col("m.sigla_uf")).alias("sigla_uf"),
        F.col("i.taxa_observada").alias("taxa_observada"),
        F.col("i.media_portugues").alias("media_portugues"),
        F.col("i.qtd_redes").alias("qtd_redes"),
        F.col("m.meta_do_ano").alias("meta_do_ano"),
        F.col("m.percentual_participacao").alias("percentual_participacao"),
        *[F.col(f"m.meta_{a}").alias(f"meta_{a}") for a in range(2024, 2031)],
    )
    .withColumn(
        "gap_pp",
        F.when(
            F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull(),
            F.round(F.col("taxa_observada") - F.col("meta_do_ano"), 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "status_meta",
        F.when(F.col("taxa_observada").isNull(), F.lit("sem_dado"))
         .when(F.col("meta_do_ano").isNull(), F.lit("sem_meta"))
         .when(F.col("taxa_observada") >= F.col("meta_do_ano"), F.lit("atingiu"))
         .otherwise(F.lit("abaixo"))
    )
    .withColumn(
        "tem_dado_completo",
        (F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull()).cast("boolean")
    )
)

print(f"Registros na Gold UF: {df_gold_uf.count()}")

print("\nDistribuição de status_meta:")
df_gold_uf.groupBy("status_meta").count().orderBy(F.desc("count")).show()

print("\nStatus por ano:")
df_gold_uf.groupBy("ano", "status_meta").count().orderBy("ano", "status_meta").show()

In [0]:
print("UFs sem_dado em 2024:")
df_gold_uf.filter((F.col("ano") == 2024) & (F.col("status_meta") == "sem_dado")) \
    .select("sigla_uf", "taxa_observada", "meta_do_ano").show()

print("\nUFs sem_meta em 2024:")
df_gold_uf.filter((F.col("ano") == 2024) & (F.col("status_meta") == "sem_meta")) \
    .select("sigla_uf", "taxa_observada", "meta_do_ano").show()

In [0]:
print("Meta de DF e RR por ano:")
df_uf_meta_prep.filter(F.col("sigla_uf").isin("DF", "RR")) \
    .select("ano", "sigla_uf", "meta_do_ano", "percentual_participacao") \
    .orderBy("sigla_uf", "ano").show()

print("\nUFs com meta nula em 2024:")
df_uf_meta_prep.filter((F.col("ano") == 2024) & F.col("meta_do_ano").isNull()) \
    .select("sigla_uf", "meta_do_ano").show()

In [0]:
# ==========================================================
# Célula 5 — Data Quality da Gold UF
#
# Autossuficiente: reconstrói df_gold_uf no topo.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"
TABELA_IND_UF  = f"{CATALOGO}.{SCHEMA}.indicador_uf"
TABELA_META_UF = f"{CATALOGO}.{SCHEMA}.meta_uf"

df_uf_ind_prep = (
    spark.table(TABELA_IND_UF)
    .groupBy("ano", "sigla_uf")
    .agg(
        F.round(F.avg("taxa_alfabetizacao"), 4).alias("taxa_observada"),
        F.round(F.avg("media_portugues"), 4).alias("media_portugues"),
        F.countDistinct("rede").alias("qtd_redes"),
        F.count("*").alias("qtd_registros_rede"),
    )
)

df_uf_meta_prep = (
    spark.table(TABELA_META_UF)
    .select(
        F.col("ano").cast("int").alias("ano"),
        F.col("sigla_uf").cast("string").alias("sigla_uf"),
        F.col("percentual_participacao").cast("double").alias("percentual_participacao"),
        *[F.col(f"meta_alfabetizacao_{a}").cast("double").alias(f"meta_{a}") for a in range(2024, 2031)],
    )
    .withColumn(
        "meta_do_ano",
        F.when(F.col("ano") == 2024, F.col("meta_2024"))
         .when(F.col("ano") == 2025, F.col("meta_2025"))
         .when(F.col("ano") == 2026, F.col("meta_2026"))
         .when(F.col("ano") == 2027, F.col("meta_2027"))
         .when(F.col("ano") == 2028, F.col("meta_2028"))
         .when(F.col("ano") == 2029, F.col("meta_2029"))
         .when(F.col("ano") == 2030, F.col("meta_2030"))
         .otherwise(F.lit(None).cast("double"))
    )
    .dropDuplicates(["ano", "sigla_uf"])
)

df_gold_uf = (
    df_uf_ind_prep.alias("i")
    .join(df_uf_meta_prep.alias("m"), on=["ano", "sigla_uf"], how="full_outer")
    .select(
        F.coalesce(F.col("i.ano"), F.col("m.ano")).alias("ano"),
        F.coalesce(F.col("i.sigla_uf"), F.col("m.sigla_uf")).alias("sigla_uf"),
        F.col("i.taxa_observada").alias("taxa_observada"),
        F.col("i.media_portugues").alias("media_portugues"),
        F.col("i.qtd_redes").alias("qtd_redes"),
        F.col("m.meta_do_ano").alias("meta_do_ano"),
        F.col("m.percentual_participacao").alias("percentual_participacao"),
        *[F.col(f"m.meta_{a}").alias(f"meta_{a}") for a in range(2024, 2031)],
    )
    .withColumn(
        "gap_pp",
        F.when(
            F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull(),
            F.round(F.col("taxa_observada") - F.col("meta_do_ano"), 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "status_meta",
        F.when(F.col("taxa_observada").isNull(), F.lit("sem_dado"))
         .when(F.col("meta_do_ano").isNull(), F.lit("sem_meta"))
         .when(F.col("taxa_observada") >= F.col("meta_do_ano"), F.lit("atingiu"))
         .otherwise(F.lit("abaixo"))
    )
    .withColumn(
        "tem_dado_completo",
        (F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull()).cast("boolean")
    )
)

# --- Validações ---
total_uf = df_gold_uf.count()

nulos_chave_uf = df_gold_uf.filter(
    F.col("ano").isNull() | F.col("sigla_uf").isNull()
).count()

duplicatas_uf = total_uf - df_gold_uf.dropDuplicates(["ano", "sigla_uf"]).count()

status_invalido_uf = df_gold_uf.filter(
    ~F.col("status_meta").isin("atingiu", "abaixo", "sem_meta", "sem_dado")
).count()

gap_inconsistente_uf = df_gold_uf.filter(
    (F.col("gap_pp").isNotNull()) &
    (F.col("taxa_observada").isNull() | F.col("meta_do_ano").isNull())
).count()

print("=" * 55)
print("DATA QUALITY — gold_indicador_uf")
print("=" * 55)
print(f"Registros totais       : {total_uf}")
print(f"Chave nula             : {nulos_chave_uf}")
print(f"Duplicatas na chave    : {duplicatas_uf}")
print(f"Status fora do domínio : {status_invalido_uf}")
print(f"Gap inconsistente      : {gap_inconsistente_uf}")
print(f"Anos                   : {[r[0] for r in df_gold_uf.select('ano').distinct().orderBy('ano').collect()]}")
print(f"UFs distintas          : {df_gold_uf.select('sigla_uf').distinct().count()}")
print("=" * 55)

assert nulos_chave_uf == 0, "FALHA: chave nula"
assert duplicatas_uf == 0, "FALHA: duplicatas na chave"
assert status_invalido_uf == 0, "FALHA: status fora do domínio"
assert gap_inconsistente_uf == 0, "FALHA: gap inconsistente"
print("Todas as regras críticas passaram.")

In [0]:
# ==========================================================
# Célula 6 — Gravação e validação da Gold UF
#
# Autossuficiente: reconstrói df_gold_uf no topo.
# O OPTIMIZE está protegido por try/except para não quebrar
# a célula em ambientes com limitação de recurso.
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"
TABELA_IND_UF  = f"{CATALOGO}.{SCHEMA}.indicador_uf"
TABELA_META_UF = f"{CATALOGO}.{SCHEMA}.meta_uf"
TABELA_DEST_UF = f"{CATALOGO}.{SCHEMA}.gold_indicador_uf"

df_uf_ind_prep = (
    spark.table(TABELA_IND_UF)
    .groupBy("ano", "sigla_uf")
    .agg(
        F.round(F.avg("taxa_alfabetizacao"), 4).alias("taxa_observada"),
        F.round(F.avg("media_portugues"), 4).alias("media_portugues"),
        F.countDistinct("rede").alias("qtd_redes"),
        F.count("*").alias("qtd_registros_rede"),
    )
)

df_uf_meta_prep = (
    spark.table(TABELA_META_UF)
    .select(
        F.col("ano").cast("int").alias("ano"),
        F.col("sigla_uf").cast("string").alias("sigla_uf"),
        F.col("percentual_participacao").cast("double").alias("percentual_participacao"),
        *[F.col(f"meta_alfabetizacao_{a}").cast("double").alias(f"meta_{a}") for a in range(2024, 2031)],
    )
    .withColumn(
        "meta_do_ano",
        F.when(F.col("ano") == 2024, F.col("meta_2024"))
         .when(F.col("ano") == 2025, F.col("meta_2025"))
         .when(F.col("ano") == 2026, F.col("meta_2026"))
         .when(F.col("ano") == 2027, F.col("meta_2027"))
         .when(F.col("ano") == 2028, F.col("meta_2028"))
         .when(F.col("ano") == 2029, F.col("meta_2029"))
         .when(F.col("ano") == 2030, F.col("meta_2030"))
         .otherwise(F.lit(None).cast("double"))
    )
    .dropDuplicates(["ano", "sigla_uf"])
)

df_gold_uf = (
    df_uf_ind_prep.alias("i")
    .join(df_uf_meta_prep.alias("m"), on=["ano", "sigla_uf"], how="full_outer")
    .select(
        F.coalesce(F.col("i.ano"), F.col("m.ano")).alias("ano"),
        F.coalesce(F.col("i.sigla_uf"), F.col("m.sigla_uf")).alias("sigla_uf"),
        F.col("i.taxa_observada").alias("taxa_observada"),
        F.col("i.media_portugues").alias("media_portugues"),
        F.col("i.qtd_redes").alias("qtd_redes"),
        F.col("m.meta_do_ano").alias("meta_do_ano"),
        F.col("m.percentual_participacao").alias("percentual_participacao"),
        *[F.col(f"m.meta_{a}").alias(f"meta_{a}") for a in range(2024, 2031)],
    )
    .withColumn(
        "gap_pp",
        F.when(
            F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull(),
            F.round(F.col("taxa_observada") - F.col("meta_do_ano"), 2)
        ).otherwise(F.lit(None).cast("double"))
    )
    .withColumn(
        "status_meta",
        F.when(F.col("taxa_observada").isNull(), F.lit("sem_dado"))
         .when(F.col("meta_do_ano").isNull(), F.lit("sem_meta"))
         .when(F.col("taxa_observada") >= F.col("meta_do_ano"), F.lit("atingiu"))
         .otherwise(F.lit("abaixo"))
    )
    .withColumn(
        "tem_dado_completo",
        (F.col("taxa_observada").isNotNull() & F.col("meta_do_ano").isNotNull()).cast("boolean")
    )
)

# --- Gravação ---
(df_gold_uf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ano")
    .saveAsTable(TABELA_DEST_UF))

print(f"Gravado em {TABELA_DEST_UF}")

df_check_uf = spark.table(TABELA_DEST_UF)
print(f"Registros: {df_check_uf.count()}")

print("\nCobertura por ano:")
df_check_uf.groupBy("ano").agg(
    F.count("*").alias("ufs"),
    F.sum(F.when(F.col("tem_dado_completo"), 1).otherwise(0)).alias("com_dado_completo"),
    F.sum(F.when(F.col("status_meta") == "atingiu", 1).otherwise(0)).alias("atingiu"),
    F.sum(F.when(F.col("status_meta") == "abaixo", 1).otherwise(0)).alias("abaixo"),
).orderBy("ano").show()

print("\nRanking de gap em 2024 (todas as UFs):")
df_check_uf.filter((F.col("ano") == 2024) & F.col("gap_pp").isNotNull()) \
    .orderBy(F.desc("gap_pp")) \
    .select("sigla_uf", "taxa_observada", "meta_do_ano", "gap_pp", "qtd_redes") \
    .show(30, truncate=False)

# --- ZORDER protegido ---
try:
    spark.sql(f"OPTIMIZE {TABELA_DEST_UF} ZORDER BY (sigla_uf)")
    print("\nZORDER aplicado em sigla_uf.")
except Exception as e:
    print(f"\nZORDER não aplicado (não bloqueia a gravação): {e}")